# 02b — Feature Extraction: LOSO Folds

Two-phase design so heavy image loading runs **once**, not 20 times:

**Phase 1** — Extract raw features for all 6,183 samples → `features/raw_all.csv`
(inf-replaced, outlier-clipped; no QT, no imputation — those are fold-dependent)

**Phase 2** — For each of 20 LOSO folds: split `raw_all.csv` by `AERONET_Site`, fit QT + variance filter + imputer on the 19-site train set, apply to val, save `features/loso/{site}/`.

No prerequisite notebooks needed — reads directly from `Data_Collection/Dataset/dataset.csv` and the GeoTIFFs.

**Output:** `features/raw_all.csv` (Phase 1) and `features/loso/{site}/train.csv`, `val.csv`, `transformers.joblib` (Phase 2).

Expected runtime: Phase 1 ≈ 30 min | Phase 2 ≈ 5 min total.

In [1]:
SKIP_PHASE1 = False  # set True if raw_all.csv already exists
RESUME      = True   # Phase 2: skip folds whose output dir already has train.csv
IMG_SIZE    = 128

In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tifffile
import cv2
import joblib
import itertools
from pathlib import Path
from datetime import datetime, timezone
from time import time
from tqdm import tqdm
from scipy.stats import kurtosis, skew, entropy
from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import QuantileTransformer
from pysolar.solar import get_azimuth

ROOT        = Path('D:/Regional-Scale-AOD-Estimation')
CSV         = ROOT / 'Data_Collection/Dataset/dataset.csv'
FEAT_OUT    = ROOT / 'feature_engineering/features/loso'
RAW_ALL     = ROOT / 'feature_engineering/features/raw_all.csv'
FEAT_OUT.mkdir(parents=True, exist_ok=True)

BAND_NAMES = ['B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B10','B11','B12']
META_COLS  = ['Day_of_Year','Site_Latitude(Degrees)','Site_Longitude(Degrees)',
              'Site_Elevation(m)','Solar_Zenith_Angle(Degrees)',
              'Solar_Azimuth_Angle(Degrees)','Airmass','Year','Month']
TARGET_COL = 'AOD_550nm'
SITE_COL   = 'AERONET_Site'
PATH_COL   = 'image_path'
EPS        = 1e-6

print(f'raw_all → {RAW_ALL}')
print(f'loso    → {FEAT_OUT}')

raw_all → D:\Regional-Scale-AOD-Estimation\feature_engineering\features\raw_all.csv
loso    → D:\Regional-Scale-AOD-Estimation\feature_engineering\features\loso


In [3]:
# ── helpers (identical to 02_features.ipynb) ──────────────────────────────

def parse_date(s):
    d, m, y = s.split(':')
    return datetime(int(y), int(m), int(d))

def compute_saa(image_paths, lats, lons):
    saas = []
    for p, lat, lon in zip(image_paths, lats, lons):
        fname = Path(p).stem
        dt    = datetime.strptime(fname, "%Y%m%d_%H%M%S").replace(tzinfo=timezone.utc)
        saas.append(get_azimuth(float(lat), float(lon), dt))
    return np.array(saas, dtype=np.float32)

def prepare_meta(df_):
    dates = df_['Date(dd:mm:yyyy)'].apply(parse_date)
    df_ = df_.copy()
    df_['Year']  = dates.dt.year
    df_['Month'] = dates.dt.month
    sza_rad = np.deg2rad(df_['Solar_Zenith_Angle(Degrees)'].to_numpy())
    df_['Airmass'] = 1.0 / np.cos(sza_rad)
    df_['Solar_Azimuth_Angle(Degrees)'] = compute_saa(
        df_['image_path'], df_['Site_Latitude(Degrees)'], df_['Site_Longitude(Degrees)']
    )
    return df_[META_COLS + [TARGET_COL, PATH_COL]].reset_index(drop=True)

def to_uint8_glcm(patch_2d):
    lo, hi = np.percentile(patch_2d, [1, 99])
    if hi - lo < EPS:
        return np.zeros_like(patch_2d, dtype=np.uint8)
    return np.clip((patch_2d - lo) / (hi - lo) * 255, 0, 255).astype(np.uint8)

def _stats(vals, prefix):
    return {
        f'{prefix}_mean'    : np.mean(vals, axis=1),
        f'{prefix}_var'     : np.var(vals, axis=1),
        f'{prefix}_median'  : np.median(vals, axis=1),
        f'{prefix}_kurtosis': kurtosis(vals, axis=1),
        f'{prefix}_skew'    : skew(vals, axis=1),
        f'{prefix}_std'     : np.std(vals, axis=1),
        f'{prefix}_p10'     : np.percentile(vals, 10, axis=1),
        f'{prefix}_p25'     : np.percentile(vals, 25, axis=1),
        f'{prefix}_p75'     : np.percentile(vals, 75, axis=1),
        f'{prefix}_p90'     : np.percentile(vals, 90, axis=1),
    }

def _glcm(vals, prefix, img_size=IMG_SIZE):
    props = ['dissimilarity', 'homogeneity', 'contrast', 'correlation', 'energy']
    cols  = {f'{prefix}_GLCM_{p}': [] for p in props}
    for i in range(vals.shape[0]):
        patch = to_uint8_glcm(vals[i].reshape(img_size, img_size))
        g     = graycomatrix(patch, distances=[1], angles=[0], symmetric=True, normed=True)
        for p in props:
            cols[f'{prefix}_GLCM_{p}'].append(graycoprops(g, p)[0, 0])
    return cols

def _index_features(index_dict, img_size=IMG_SIZE):
    new_cols, glcm_cols = {}, {}
    for name, vals in tqdm(index_dict.items(), desc='  indices', leave=False):
        new_cols.update(_stats(vals, name))
        glcm_cols.update(_glcm(vals, name, img_size))
    return pd.DataFrame({**new_cols, **glcm_cols})

def load_bands(paths, img_size=IMG_SIZE, min_coverage=0.90, cirrus_p95_thresh=300.0):
    sqrt_out, raw_out, kept_idx = [], [], []
    n_drop_cov = 0; n_drop_cir = 0
    for idx, p in enumerate(tqdm(paths, desc='loading', leave=False)):
        img = tifffile.imread(str(ROOT / p)).astype(np.float32)
        img = np.transpose(img, (2, 0, 1))
        resized = np.stack([
            cv2.resize(img[b], (img_size, img_size), interpolation=cv2.INTER_AREA)
            for b in range(img.shape[0])
        ], axis=0)
        n_pixels  = img_size * img_size
        good_mask = (resized > 0) & (resized <= 10000)
        if good_mask.sum(axis=(1,2)).min() / n_pixels < min_coverage:
            n_drop_cov += 1; continue
        b10_good = resized[10][good_mask[10]]
        if b10_good.size == 0 or np.percentile(b10_good, 95) > cirrus_p95_thresh:
            n_drop_cir += 1; continue
        for b in range(resized.shape[0]):
            bad = ~good_mask[b]
            if bad.any():
                good_vals = resized[b][good_mask[b]]
                resized[b][bad] = good_vals.mean() if good_vals.size > 0 else 0.0
        sqrt_out.append(np.sqrt(resized).reshape(13, -1))
        raw_out.append((resized / 10000.0).reshape(13, -1))
        kept_idx.append(idx)
    print(f'    dropped coverage:{n_drop_cov} cirrus:{n_drop_cir} kept:{len(kept_idx)}')
    return (np.asarray(sqrt_out, dtype=np.float32),
            np.asarray(raw_out,  dtype=np.float32),
            kept_idx)

def calculate_statistics(bands):
    new_cols = {}
    for j, name in enumerate(tqdm(BAND_NAMES, desc='band stats', leave=False)):
        new_cols.update(_stats(bands[:, j, :], name))
    pairs = list(itertools.combinations(range(len(BAND_NAMES)), 2))
    for (i, j) in tqdm(pairs, desc='band ratios', leave=False):
        Ba, Bb = bands[:, i, :], bands[:, j, :]
        new_cols.update(_stats((Ba - Bb) / (Ba + Bb + EPS), f'rdiff_{i}_{j}'))
        new_cols.update(_stats(Ba / (Bb + EPS),             f'rdiv_{i}_{j}'))
    return pd.DataFrame(new_cols)

def calculate_advanced_band_features(bands):
    new_cols = {}
    pairs   = list(itertools.combinations(range(len(BAND_NAMES)), 2))
    triples = list(itertools.combinations(range(len(BAND_NAMES)), 3))
    for (i, j) in tqdm(pairs, desc='adv pairs', leave=False):
        Ba, Bb = bands[:, i, :], bands[:, j, :]
        new_cols[f'geomean_{i}_{j}']   = np.sqrt(Ba * Bb).mean(axis=1)
        new_cols[f'harmmean_{i}_{j}']  = (2*Ba*Bb / (Ba+Bb+EPS)).mean(axis=1)
        new_cols[f'normratio_{i}_{j}'] = ((Ba-Bb) / (np.sqrt(Ba**2+Bb**2)+EPS)).mean(axis=1)
    for i, name in tqdm(enumerate(BAND_NAMES), desc='entropy', total=len(BAND_NAMES), leave=False):
        new_cols[f'{name}_entropy'] = np.apply_along_axis(
            lambda x: entropy(np.histogram(x, bins=256, density=True)[0] + EPS), 1, bands[:, i, :])
    for (i, j, k) in tqdm(triples, desc='triples', leave=False):
        Ba, Bb, Bc = bands[:, i, :], bands[:, j, :], bands[:, k, :]
        new_cols[f'dprod_{i}_{j}_{k}']  = ((Ba*Bb) - (Ba*Bc)).mean(axis=1)
        new_cols[f'cratio_{i}_{j}_{k}'] = ((Ba*Bb) / (Bc+EPS)).mean(axis=1)
    return pd.DataFrame(new_cols)

def compute_texture_features(bands_raw, img_size=IMG_SIZE):
    B02 = bands_raw[:, 1, :]
    scale_max = 0.5
    props = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']
    cols  = {f'B02raw_GLCM_{p}': [] for p in props}
    for i in tqdm(range(B02.shape[0]), desc='GLCM B02', leave=False):
        patch = np.clip(B02[i], 0.0, scale_max) / scale_max
        patch = (patch * 255.0).astype(np.uint8).reshape(img_size, img_size)
        g = graycomatrix(patch, distances=[1],
                         angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                         levels=256, symmetric=True, normed=True)
        for p in props:
            cols[f'B02raw_GLCM_{p}'].append(graycoprops(g, p).mean())
    return pd.DataFrame(cols)

def calculate_vegetation_indices(bands):
    B01=bands[:,0,:]; B02=bands[:,1,:]; B03=bands[:,2,:]; B04=bands[:,3,:]
    B05=bands[:,4,:]; B06=bands[:,5,:]; B07=bands[:,6,:]; B08=bands[:,7,:]
    B8A=bands[:,8,:]; B09=bands[:,9,:]; B10=bands[:,10,:]; B11=bands[:,11,:]; B12=bands[:,12,:]
    indices = {
        'ndvi'  : (B08-B04)/(B08+B04+EPS), 'gli':(2*B03-B04-B02)/(2*B03+B04+B02+EPS),
        'cvi'   : (B08*B04)/(B03**2+EPS),  'sipi':(B08-B02)/(B08-B04+EPS),
        'ndwi'  : (B08-B11)/(B08+B11+EPS), 'ccci':((B08-B05)/(B08+B05+EPS))/((B08-B04)/(B08+B04)+EPS),
        'hue'   : np.arctan2((B02-B03),(B03-B04)), 'rendvi':(B06-B05)/(B06+B05+EPS),
        'reci'  : (B08/(B04+EPS))-1,       'evi':2.5*(B08-B04)/(B08+6*B04-7.5*B02+1.0+EPS),
        'evi2'  : 2.4*(B08-B04)/(B08+B04+1.0+EPS), 'bsi':(B11+B04)/(B08+B02+EPS),
        'npcri' : (B03-B08)/(B03+B08+EPS), 'mndvi':(B08-B04)/(B08+B04-2*B02+EPS),
        'ndsi'  : (B03-B11)/(B03+B11+EPS), 'ndmi':(B08-B11)/(B08+B11+EPS),
        'ndbi'  : (B11-B08)/(B11+B08+EPS), 'ndci':(B05-B04)/(B05+B04+EPS),
        'gndvi' : (B08-B03)/(B08+B03+EPS), 'savi':((B01-B02)/(B01+B02+0.428+EPS))*1.428,
    }
    return _index_features(indices)

def calculate_vegetation_indices2(bands):
    B01=bands[:,0,:]; B02=bands[:,1,:]; B03=bands[:,2,:]; B04=bands[:,3,:]
    B05=bands[:,4,:]; B06=bands[:,5,:]; B07=bands[:,6,:]; B08=bands[:,7,:]
    B8A=bands[:,8,:]; B09=bands[:,9,:]; B10=bands[:,10,:]; B11=bands[:,11,:]; B12=bands[:,12,:]
    indices = {
        'ndvi_r':(B08-B07)/(B08+B07+EPS), 'chl':(B07/(B05+EPS))-1,
        'bi'    :(B04**2+B03**2+B02**2)/3, 'si':(B04-B02)/(B04+B02+EPS),
        'nmdi'  :(B08-(B11-B12))/(B08+(B11-B12)+EPS), 'wbi':B8A/(B09+EPS),
        'msi'   :B11/(B08+EPS), 'bsi1':((B12+B04)-(B8A+B02))/((B12+B04)+(B8A+B02)+EPS),
        'bsi2'  :((B11+B04)-(B8A+B02))/((B11+B04)+(B8A+B02)+EPS),
        'ndsi1' :(B11-B8A)/(B11+B8A+EPS), 'ndsi2':(B12-B03)/(B12+B03+EPS),
        'bi2'   :B04+B11-B8A, 'dbsi':(B12-B03)/(B12+B03+EPS)-(B8A-B04)/(B8A+B04+EPS),
        'mbi'   :((B11+B12+B8A)/(B11+B12+B8A+EPS))+0.5, 'r01':B01/(B03+EPS),
        'r02'   :B01/(B05+EPS), 'r03':B11/(B12+EPS), 'r04':B05/(B04+EPS),
        'mi'    :(B8A-B11)/(B8A+B11+EPS), 'mresr':(B06-B01)/(B05-B01+EPS),
        'psri'  :(B04-B02)/(B06+EPS), 'tvi':(120*(B06-B03)-200*(B04-B03))/2,
        'arvi'  :(B08-2*B04+B02)/(B08+2*B04+B02+EPS), 'exg':2*B03-B04-B02,
        'aci'   :B08*(B04+B03),
    }
    return _index_features(indices)

def calculate_bloom_indices(bands):
    B02=bands[:,1,:]; B03=bands[:,2,:]; B04=bands[:,3,:]; B08=bands[:,7,:]
    NDGI=(B04-B03)/(B04+B03+EPS); DYI=B04/(B03+EPS)
    NDPI=(0.5*(B04+B02)-B03)/(0.5*(B04+B02)+B03+EPS)
    NDYI=(0.5*(B04+B03)-B02)/(0.5*(B04+B03)+B02+EPS)
    PEBI=NDPI/((NDGI+1)*B08+EPS); YEBI=NDYI/((NDGI+1)*B08+EPS)
    return _index_features({'NDGI':NDGI,'DYI':DYI,'NDPI':NDPI,'PEBI':PEBI,'NDYI':NDYI,'YEBI':YEBI})

def calculate_rededge_indices(bands):
    B03=bands[:,2,:]; B04=bands[:,3,:]; B05=bands[:,4,:]
    B06=bands[:,5,:]; B07=bands[:,6,:]; B08=bands[:,7,:]
    indices = {
        'NDVIre1':(B08-B05)/(B08+B05+EPS),'NDVIre2':(B08-B06)/(B08+B06+EPS),
        'NDVIre3':(B08-B07)/(B08+B07+EPS),'NDRE1':(B06-B05)/(B06+B05+EPS),
        'NDRE2':(B07-B05)/(B07+B05+EPS),'NDRE3':(B07-B06)/(B07+B06+EPS),
        'CIre1':(B08/(B05+EPS))-1,'CIre2':(B08/(B06+EPS))-1,'CIre3':(B08/(B07+EPS))-1,
        'MCARI1':((B05-B04)-0.2*(B05-B03))*(B05/(B04+EPS)),
        'MCARI2':((B06-B04)-0.2*(B06-B03))*(B06/(B04+EPS)),
        'MCARI3':((B07-B04)-0.2*(B07-B03))*(B07/(B04+EPS)),
        'TCARI1':3*((B05-B04)-0.2*(B05-B03))*(B05/(B04+EPS)),
        'TCARI2':3*((B06-B04)-0.2*(B06-B03))*(B06/(B04+EPS)),
        'TCARI3':3*((B07-B04)-0.2*(B07-B03))*(B07/(B04+EPS)),
        'MTCI1':(B06-B05)/(B05-B04+EPS),'MTCI2':(B07-B05)/(B05-B04+EPS),
        'MTCI3':(B07-B06)/(B06-B04+EPS),
    }
    return _index_features(indices)

def calculate_aod_indices(bands_raw, sza_deg):
    B01=bands_raw[:,0,:]; B02=bands_raw[:,1,:]; B03=bands_raw[:,2,:]; B04=bands_raw[:,3,:]
    B08=bands_raw[:,7,:]; B11=bands_raw[:,11,:]; B12=bands_raw[:,12,:]
    indices = {
        'aod_B01_over_B11':B01/(B11+EPS),'aod_B02_over_B11':B02/(B11+EPS),
        'aod_B02_over_B12':B02/(B12+EPS),'aod_nd_B02_B11':(B02-B11)/(B02+B11+EPS),
        'aod_nd_B01_B12':(B01-B12)/(B01+B12+EPS),'aod_B01_over_B02':B01/(B02+EPS),
        'aod_B04_over_B12':B04/(B12+EPS),
        'aod_spectral_curv_B02':np.log(B01+EPS)+np.log(B04+EPS)-2*np.log(B02+EPS),
        'aod_AFRI1600':(B08-0.66*B11)/(B08+0.66*B11+EPS),
        'aod_NDVI_minus_AFRI':((B08-B04)/(B08+B04+EPS)-(B08-0.66*B11)/(B08+0.66*B11+EPS)),
        'aod_dust_B04_minus_B02':B04-B02,
        'aod_dust_nd_B04_B11':(B04-B11)/(B04+B11+EPS),
        'aod_B02_airmass':B02*(1.0/np.cos(np.deg2rad(sza_deg)))[:,None],
        'aod_log_B02_airmass':np.log(B02+EPS)*(1.0/np.cos(np.deg2rad(sza_deg)))[:,None],
    }
    wl=np.array([443.0,490.0,560.0,665.0],dtype=np.float32)
    log_wl_c=np.log(wl)-np.log(wl).mean(); denom=(log_wl_c**2).sum()
    stacked=np.stack([B01,B02,B03,B04],axis=1)
    log_rho=np.log(np.clip(stacked,EPS,None))
    log_rho_c=log_rho-log_rho.mean(axis=1,keepdims=True)
    indices['aod_angstrom_alpha']=(log_wl_c[None,:,None]*log_rho_c).sum(axis=1)/denom
    new_cols = {}
    for name, vals in tqdm(indices.items(), desc='aod idx', leave=False):
        new_cols.update(_stats(vals, name))
    return pd.DataFrame(new_cols)

def extract_all_features(meta_df, bands_sqrt, bands_raw, sza_deg):
    return pd.concat([
        meta_df.drop(columns=[PATH_COL]).reset_index(drop=True),
        calculate_statistics(bands_sqrt),
        calculate_advanced_band_features(bands_sqrt),
        compute_texture_features(bands_raw),
        calculate_vegetation_indices(bands_raw),
        calculate_vegetation_indices2(bands_raw),
        calculate_bloom_indices(bands_raw),
        calculate_rededge_indices(bands_raw),
        calculate_aod_indices(bands_raw, sza_deg),
    ], axis=1)

def postprocess_and_save(train_features, val_features, out_dir):
    """Apply the same post-processing pipeline as 02_features.ipynb and save."""
    out_dir.mkdir(parents=True, exist_ok=True)

    train_y  = train_features[TARGET_COL].values
    val_y    = val_features[TARGET_COL].values
    feat_cols = [c for c in train_features.columns if c != TARGET_COL]

    extra = [c for c in val_features.columns if c not in train_features.columns and c != TARGET_COL]
    assert not extra, f'val has unexpected columns: {extra[:5]}'

    train_X = train_features[feat_cols].copy()
    val_X   = val_features[feat_cols].copy()

    train_X.replace([np.inf, -np.inf], np.nan, inplace=True)
    val_X.replace([np.inf, -np.inf],   np.nan, inplace=True)

    train_X = train_X.mask(train_X.abs() > 1e4)
    val_X   = val_X.mask(val_X.abs() > 1e4)

    thresh    = 0.2 * len(train_X)
    keep_cols = train_X.columns[train_X.notna().sum() >= thresh].tolist()
    train_X   = train_X[keep_cols]
    val_X     = val_X[keep_cols]

    train_means = train_X.mean().fillna(0.0)
    train_X = train_X.fillna(train_means)
    val_X   = val_X.fillna(train_means)

    train_X_arr = train_X.to_numpy(dtype=np.float64)
    val_X_arr   = val_X.to_numpy(dtype=np.float64)

    meta_feat_cols = [c for c in keep_cols if c in META_COLS]
    img_feat_cols  = [c for c in keep_cols if c not in META_COLS]
    meta_idx = np.array([keep_cols.index(c) for c in meta_feat_cols])
    img_idx  = np.array([keep_cols.index(c) for c in img_feat_cols])

    img_stds      = train_X_arr[:, img_idx].std(axis=0)
    keep_img_mask = img_stds > 1e-5
    kept_img_cols = [c for c, k in zip(img_feat_cols, keep_img_mask) if k]
    kept_img_idx  = img_idx[keep_img_mask]

    qt = QuantileTransformer(output_distribution='normal', random_state=42)
    train_X_arr[:, kept_img_idx] = qt.fit_transform(train_X_arr[:, kept_img_idx])
    val_X_arr[:, kept_img_idx]   = qt.transform(val_X_arr[:, kept_img_idx])

    final_cols    = meta_feat_cols + kept_img_cols
    all_final_idx = np.concatenate([meta_idx, kept_img_idx])

    train_out = pd.DataFrame(train_X_arr[:, all_final_idx], columns=final_cols)
    train_out.insert(0, TARGET_COL, train_y)
    val_out   = pd.DataFrame(val_X_arr[:, all_final_idx], columns=final_cols)
    val_out.insert(0, TARGET_COL, val_y)

    # QT on meta cols
    saved_meta_cols = [c for c in META_COLS if c in train_out.columns]
    qt_meta = QuantileTransformer(output_distribution='normal', random_state=42)
    train_out[saved_meta_cols] = qt_meta.fit_transform(train_out[saved_meta_cols])
    val_out[saved_meta_cols]   = qt_meta.transform(val_out[saved_meta_cols])

    train_out.to_csv(out_dir / 'train.csv', index=False)
    val_out.to_csv(out_dir   / 'val.csv',   index=False)

    joblib.dump({
        'qt': qt, 'qt_meta': qt_meta,
        'keep_cols': keep_cols, 'train_means': train_means,
        'kept_img_cols': kept_img_cols, 'meta_feat_cols': saved_meta_cols,
        'final_cols': final_cols,
    }, out_dir / 'transformers.joblib')

    return train_out.shape, val_out.shape

print('All helper functions loaded')

All helper functions loaded


## Phase 1 — Extract raw features for all samples (run once)

In [4]:
if SKIP_PHASE1:
    print('Phase 1 skipped — loading existing raw_all.csv')
    _check = pd.read_csv(RAW_ALL, nrows=2)
    print(f'  raw_all.csv exists: {RAW_ALL}')
else:
    t0 = time()

    all_df = pd.read_csv(CSV)
    all_df = all_df.drop(columns=['Time(hh:mm:ss)', 'Day_of_Year(Fraction)'])
    print(f'Total samples in dataset.csv: {len(all_df)}')

    all_meta = prepare_meta(all_df)
    site_series = all_df[SITE_COL].reset_index(drop=True)

    print('\nLoading all images...')
    all_sqrt, all_raw, kept_idx = load_bands(all_meta[PATH_COL])
    all_meta_kept = all_meta.iloc[kept_idx].reset_index(drop=True)
    site_kept     = site_series.iloc[kept_idx].reset_index(drop=True)
    print(f'Samples after image QC: {len(kept_idx)} / {len(all_df)}')

    print('\nExtracting features...')
    all_feats = extract_all_features(
        all_meta_kept, all_sqrt, all_raw,
        all_meta_kept['Solar_Zenith_Angle(Degrees)'].to_numpy()
    )

    all_feats.insert(0, SITE_COL, site_kept.values)

    # Replace inf and clip extreme outliers — fold-agnostic pre-processing.
    # Exclude SITE_COL (string) and TARGET_COL (AOD 0-3, never hits 1e4) from mask
    # to match 02_features.ipynb semantics exactly.
    clip_cols = [c for c in all_feats.select_dtypes(include=[np.number]).columns
                 if c != TARGET_COL]
    all_feats.replace([np.inf, -np.inf], np.nan, inplace=True)
    all_feats[clip_cols] = all_feats[clip_cols].mask(all_feats[clip_cols].abs() > 1e4)

    all_feats.to_csv(RAW_ALL, index=False)
    print(f'\nSaved raw_all.csv: {all_feats.shape}  ({(time()-t0)/60:.1f} min)')
    print(f'Sites: {all_feats[SITE_COL].nunique()}  |  AOD range: '
          f'{all_feats[TARGET_COL].min():.3f} – {all_feats[TARGET_COL].max():.3f}')

Total samples in dataset.csv: 6183

Loading all images...


    dropped coverage:975 cirrus:74 kept:5134
Samples after image QC: 5134 / 6183

Extracting features...



Saved raw_all.csv: (5134, 3710)  (44.4 min)
Sites: 20  |  AOD range: 0.008 – 2.928


## Phase 2 — LOSO postprocessing loop (20× fast)

In [5]:
raw_all = pd.read_csv(RAW_ALL)
sites   = sorted(raw_all[SITE_COL].unique())
print(f'raw_all: {raw_all.shape}  |  {len(sites)} sites')

t_global = time()
completed, skipped = [], []

for fold_idx, site in enumerate(sites):
    out_dir = FEAT_OUT / site

    if RESUME and (out_dir / 'train.csv').exists():
        print(f'[{fold_idx+1:2d}/{len(sites)}] {site} — skipping (already done)')
        skipped.append(site)
        continue

    t_fold = time()
    train_raw = raw_all[raw_all[SITE_COL] != site].drop(columns=SITE_COL).reset_index(drop=True)
    val_raw   = raw_all[raw_all[SITE_COL] == site].drop(columns=SITE_COL).reset_index(drop=True)

    tr_shape, va_shape = postprocess_and_save(train_raw, val_raw, out_dir)
    elapsed = time() - t_fold
    print(f'[{fold_idx+1:2d}/{len(sites)}] {site:<30}  '
          f'train={tr_shape}  val={va_shape}  ({elapsed:.1f}s)')
    completed.append(site)

print(f'\nDone. Completed: {len(completed)}  Skipped: {len(skipped)}')
print(f'Total Phase 2 time: {(time()-t_global)/60:.1f} min')

raw_all: (5134, 3710)  |  20 sites
[ 1/20] AgiaMarina_Xyliatou             train=(4672, 3702)  val=(462, 3702)  (21.8s)
[ 2/20] CUT-TEPAK                       train=(4318, 3700)  val=(816, 3700)  (20.7s)
[ 3/20] Cairo_EMA_2                     train=(4821, 3702)  val=(313, 3702)  (21.0s)
[ 4/20] DEWA_ResearchCentre             train=(5068, 3701)  val=(66, 3701)  (20.7s)
[ 5/20] Eilat                           train=(4981, 3701)  val=(153, 3701)  (20.7s)
[ 6/20] El_Farafra                      train=(5089, 3701)  val=(45, 3701)  (21.0s)
[ 7/20] IMS-METU-ERDEMLI                train=(4892, 3701)  val=(242, 3701)  (20.7s)
[ 8/20] KAUST_Campus                    train=(4858, 3702)  val=(276, 3702)  (21.0s)
[ 9/20] Kuwait_University               train=(4717, 3702)  val=(417, 3702)  (20.8s)
[10/20] Masdar_Institute                train=(4764, 3702)  val=(370, 3702)  (20.7s)
[11/20] Mezaira                         train=(4915, 3702)  val=(219, 3702)  (20.8s)
[12/20] Migal                   